In [ ]:
%load_ext autoreload
%autoreload 3

import scripts.analysis as da
import scripts.dataset_parser as dataset_parser

da.configure_plot_style()


DATA RELOAD:

In [ ]:
dataset_parser.main()
df = da.load_clean_data()
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head()

## 7. Input Data - Token Limits for LLMs

Token payload of entire docs.

In [ ]:
from pathlib import Path

DOC_BASE = Path("../data/raw/ipp_13_to_24/ipp_docs")
token_df = dataset_parser.parse_document_tokens(df, doc_base=DOC_BASE, n_samples_per_type=50)

In [ ]:
ax = da.visualise_token_limits(token_df)
da.summarise_token_limits(token_df)

## 8. SHORT

Get thresholds and ratios to use.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf

DOC_BASE = Path("../data/raw/ipp_13_to_24/ipp_docs")

short_df = (
    df.query("code == 'SHORT'")[["id", "source_file", "doc_type"]]
    .drop_duplicates(["id", "source_file"])
    .reset_index(drop=True)
)

print(f"{len(short_df)} 'SHORT'")

records = []
for student_id, source_file, doc_type in short_df.itertuples(index=False):
    cohort = Path(source_file).stem
    f = DOC_BASE / cohort / f"{student_id}.{doc_type}"

    if not f.exists():
        continue

    page_count = np.nan 

    try:
        match doc_type:
            case "pdf":
                with pymupdf.open(f) as doc:
                    page_count = doc.page_count
                    text = "".join(page.get_text() for page in doc)
            case "md":
                text = f.read_text(errors="replace")
            case _:
                continue

        words = text.split()
        word_count = len(words)
        char_count = len(text)
        
        avg_word_length = (char_count / word_count) if word_count > 0 else 0
        
        structure_count = text.count("\n\n")

        records.append({
            "id": student_id,
            "cohort": cohort,
            "fmt": doc_type,
            "page_count": page_count,
            "word_count": word_count,
            "char_count": char_count,
            "avg_word_length": round(avg_word_length, 2),
            "structure_count": structure_count,
        })
    except Exception:
        continue

wc_df = pd.DataFrame(records)
print(f"\nSuccessfully parsed {len(wc_df)} files.")

metrics = ["page_count", "word_count", "char_count", "avg_word_length", "structure_count"]
display(wc_df[metrics].describe(percentiles=[.1, .25, .5, .75, .9]).round(1))

## 9. Validation

In [ ]:
import subprocess
from pathlib import Path

import pandas as pd

DOC_BASE = Path("../data/raw/ipp_13_to_24/ipp_docs")

# Select 3 random PDFs
pdf_docs = df[df["doc_type"] == "pdf"].drop_duplicates(subset=["id", "source_file"])
valid_files = []
target_ids = []

for row in pdf_docs.sample(frac=1).itertuples():
    cohort = Path(row.source_file).stem
    f = DOC_BASE / cohort / f"{row.id}.{row.doc_type}"
    if f.exists():
        valid_files.append(str(f.resolve()))
        target_ids.append(row.id)
    if len(valid_files) >= 3:
        break

print(f"Randomly selected 3 documents: {target_ids}")

# Run the tool on these 3 files
tool_output_path = Path("../out/random_demo.csv")
tool_output_path.parent.mkdir(exist_ok=True)

cmd = ["python", "-m", "src", *valid_files, "--csv-out", str(tool_output_path.resolve())]
print("Running doc-grader tool...")
result = subprocess.run(cmd, cwd="..", capture_output=True, text=True)

if result.returncode != 0:
    print("Tool execution failed!")
    print(result.stderr)
elif not tool_output_path.exists():
    print("Tool did not generate an output CSV. It might have crashed or found no files.")
else:
    # Load tool output
    tool_df = pd.read_csv(tool_output_path)
    
    # Filter to relevant findings
    tool_findings = tool_df[["id", "code", "severity", "comment"]].rename(
        columns={"code": "tool_flag", "severity": "tool_severity", "comment": "tool_reason"}
    )
    
    # Grab the human findings for these 3 docs
    human_findings = df[df["id"].isin(target_ids)][["id", "code", "impact", "comment"]].rename(
        columns={"code": "human_flag", "impact": "human_impact", "comment": "human_comment"}
    )
    
    # Outer join to align the tool findings with the human grader findings based on the flag
    comparison_df = pd.merge(
        tool_findings, 
        human_findings, 
        left_on=["id", "tool_flag"], 
        right_on=["id", "human_flag"], 
        how="outer"
    )
    comparison_df["match"] = (~comparison_df["tool_flag"].isna()) & (~comparison_df["human_flag"].isna())
    comparison_df["detected_code"] = comparison_df["tool_flag"].combine_first(comparison_df["human_flag"])
    comparison_df = comparison_df[["id", "detected_code", "match", "tool_severity", "human_impact", "tool_reason", "human_comment"]]
    
    print(f"Found {len(tool_findings)} tool detections vs {len(human_findings)} human detections.")
    display(comparison_df.sort_values(by=["id", "match"], ascending=[True, False]).style.format(na_rep='--'))